In [6]:
# ============================================================
# CELL 1 — Imports & config
# ============================================================
import os
import json
from openai import OpenAI
from src.climber_profile import ClimberProfile, Injury

client = OpenAI(
    api_key="cglowaxE1vSmixJcBun7lKmi71qsw79E",  # from https://console.mistral.ai/
    base_url="https://api.mistral.ai/v1",
)
MODEL = "mistral-small-latest"

In [ ]:
INTERVIEW_SYSTEM_PROMPT = """
Tu es un coach escalade bienveillant et expérimenté qui réalise un entretien 
d'onboarding avec un nouveau grimpeur. Ton objectif est de collecter 
suffisamment d'informations pour construire son profil de coaching personnalisé.

Tu dois couvrir progressivement ces thèmes, dans un ordre naturel :
1. Profil physique (âge, taille, envergure, poids) — demande-les ensemble de façon légère
2. Historique de grimpe (depuis combien de temps, comment il a commencé)
3. Niveau actuel (grade redpoint, grade flash)
4. Styles préférés et points forts ressentis
5. Points faibles ressentis ou identifiés
6. Entraînement actuel (séances/semaine, durée, setup maison, autres activités)
7. Blessures actuelles ou passées importantes
8. Objectifs court terme et long terme

Règles importantes :
- Pose UNE seule question à la fois, ou un groupe logique de 2-3 questions courtes
- Reformule et valide ce que le grimpeur te dit avant de passer au thème suivant
- Adapte ton vocabulaire au niveau détecté (ne parle pas de "dévié" à quelqu'un qui grimpe depuis 6 mois)
- Si une réponse est vague, creuse avec une question de suivi
- Reste conversationnel, pas exhaustif — mieux vaut un profil partiel honnête qu'un profil complet inventé
- Quand tu estimes avoir couvert les thèmes essentiels, termine par : 
  "J'ai maintenant une bonne image de ton profil. Veux-tu ajouter autre chose avant que je le finalise ?"
"""

str

In [8]:
EXTRACTION_SYSTEM_PROMPT = """
Tu es un extracteur de données structurées. On te donne la transcription 
d'un entretien entre un coach escalade et un grimpeur.

Ton unique rôle est d'extraire les informations mentionnées et de les retourner 
en JSON pur, sans aucun texte avant ou après.

Retourne UNIQUEMENT un objet JSON valide avec les champs suivants 
(omets les champs non mentionnés, ne les invente jamais) :

{
  "name": string,
  "age": int,
  "height_cm": int,
  "wingspan_cm": int,
  "weight_kg": float,
  "years_climbing": float,
  "started_at_grade": string,          // grade Fontainebleau, ex: "5b"
  "current_redpoint_grade": string,    // ex: "7a"
  "current_flash_grade": string,       // ex: "6b+"
  "preferred_styles": [string],        // ex: ["dynamique", "compression"]
  "self_strengths": [string],
  "self_weaknesses": [string],
  "gym_sessions_per_week": int,
  "typical_session_duration_min": int,
  "other_activities": [string],
  "home_setup": [string],              // ex: ["pan", "fingerboard"]
  "injuries": [
    {
      "description": string,
      "active": bool,
      "avoid": [string]
    }
  ],
  "short_term_goals": [string],
  "long_term_goals": [string],
  "coach_tone": string,                // "direct" | "encouraging" | "analytical" | "concise"
  "coach_language": "fr",
  "focus_preference": string           // "weaknesses" | "strengths" | "balanced"
}
"""

In [13]:
def run_interview() -> list[dict]:
    """
    messages only contains the conversation history passed to the API.
    Mistral requires it to always end with a user message on each call.
    """
    messages = []

    print("=== Entretien de profil grimpeur ===")
    print("(tapez 'fin' pour terminer)\n")

    # Opening: seed with the first user message, get assistant reply
    messages.append({
        "role": "user",
        "content": "Bonjour, je voudrais créer mon profil de coaching."
    })
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": INTERVIEW_SYSTEM_PROMPT}] + messages,
    )
    opening_text = response.choices[0].message.content
    messages.append({"role": "assistant", "content": opening_text})
    print(f"Coach : {opening_text}\n")

    # Conversation loop — always append user first, then call, then append assistant
    while True:
        user_input = input("Toi : ").strip()
        if not user_input:
            continue
        if user_input.lower() == "fin":
            print("\n[Entretien terminé]")
            break

        # Append user turn BEFORE the API call → history ends with user ✓
        messages.append({"role": "user", "content": user_input})

        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "system", "content": INTERVIEW_SYSTEM_PROMPT}] + messages,
        )
        assistant_text = response.choices[0].message.content
        messages.append({"role": "assistant", "content": assistant_text})
        print(f"\nCoach : {assistant_text}\n")

    return messages

In [14]:
def extract_profile(messages: list[dict]) -> ClimberProfile:
    """
    Takes the full interview transcript and runs a second LLM call
    to extract the structured ClimberProfile from it.
    """
    # Build a readable transcript string
    transcript = "\n\n".join(
        f"{'Coach' if m['role'] == 'assistant' else 'Grimpeur'} : {m['content']}"
        for m in messages
    )

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": EXTRACTION_SYSTEM_PROMPT},
            {"role": "user",   "content": f"Transcription :\n\n{transcript}"},
        ]
    )
    raw = response.choices[0].message.content.strip()

    # Strip markdown code fences if the model added any
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    raw = raw.strip()

    data = json.loads(raw)
    
    # Build ClimberProfile from extracted dict
    # injuries need special handling since they're nested objects
    injuries_data = data.pop("injuries", [])
    
    profile = ClimberProfile.from_dict({**data, "injuries": injuries_data})
    return profile

In [15]:
# Step 1: run the interview
messages = run_interview()

# Step 2: extract the structured profile
print("\n[Extraction du profil en cours...]\n")
profile = extract_profile(messages)

# Step 3: preview
print(profile.to_llm_context())

# Step 4: save
profile.save("franck.json")
print(f"\nProfil sauvegardé → franck.json")

=== Entretien de profil grimpeur ===
(tapez 'fin' pour terminer)

Coach : Bonjour et bienvenue ! Je suis ravi de t’accompagner dans cette aventure. Pour commencer, peux-tu me dire quelques mots sur toi : ton âge, ta taille, ton envergure (la distance entre tes bras tendus) et ton poids ? C’est juste pour avoir une idée de ton profil physique de base.


Coach : Merci pour ces infos ! Donc tu as 25 ans, tu mesures 170 cm avec une envergure de 178 cm (ce qui est un ratio plutôt favorable pour la grimpe) et tu pèses environ 60 kg. C’est noté.

Maintenant, parlons de ton historique : depuis combien de temps grimpes-tu, et comment as-tu commencé ?


Coach : D’accord, donc tu as 2 ans de pratique, tous en bloc au début. C’est un bon point de départ pour développer de la force et de la technique.

Est-ce que tu as continué uniquement en bloc, ou tu as aussi essayé la voie (en mur) depuis ? Et si oui, depuis combien de temps ?


Coach : Merci pour la précision ! Donc tu as 2 ans de pratique, av